# Week 42, Day 1: Hebbian learning: correlations, competition and ocular dominance

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w042/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Plain Hebb explodes

Simulate the averaged plain Hebbian rule, w += eta C w, for 2,000 steps from w0, recording the norm of w at each step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1989); n = 10
C = np.full((2 * n, 2 * n), 0.2); C[:n, :n] = 0.5; C[n:, n:] = 0.5; np.fill_diagonal(C, 1.0)
w0 = 0.5 + 0.05 * r.normal(size=2 * n)
eta = 0.01

w_hebb = w0.copy(); norms = []
for _ in range(2000):
    ...
    norms.append(np.linalg.norm(w_hebb))
print(f"weight norm: {norms[0]:.2f} at the start, {norms[-1]:.3g} after 2,000 steps")
plt.semilogy(norms); plt.xlabel("step"); plt.ylabel("|w|"); plt.show()

In [ ]:
import numpy as np
def test_explode():
    "the weight norm grows by more than a factor of 1000"
    assert norms[-1] / norms[0] > 1e3

def test_direction():
    "the weights align with the top eigenvector of C"
    ev, E = np.linalg.eigh(C)
    top = E[:, -1]
    assert abs(np.dot(w_hebb, top)) / np.linalg.norm(w_hebb) > 0.999

test_explode()
test_direction()
print("All checks passed")

### Exercise: Subtractive against multiplicative

Implement both rules in their averaged forms, each for 3,000 steps from w0: subtractive (dw = C w minus its mean, then clip to [0, 1]) and Oja (dw = C w - (wᵀ C w) w).

In [ ]:
w_sub = w0.copy(); w_oja = w0.copy()
for _ in range(3000):
    ...
print(f"subtractive: left eye {w_sub[:n].mean():.2f}, right eye {w_sub[n:].mean():.2f}")
print(f"Oja:         left eye {w_oja[:n].mean():.2f}, right eye {w_oja[n:].mean():.2f}")
plt.bar(np.arange(2 * n) - 0.2, w_sub, 0.4, label="subtractive"); plt.bar(np.arange(2 * n) + 0.2, w_oja, 0.4, label="Oja")
plt.axvline(n - 0.5, color="k", lw=0.5); plt.xlabel("input (left eye | right eye)"); plt.ylabel("weight"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_subtractive():
    "subtractive normalisation drives one eye's weights to 1 and the other's to 0"
    a, b = w_sub[:n].mean(), w_sub[n:].mean()
    assert max(a, b) > 0.95 and min(a, b) < 0.05

def test_oja():
    "Oja's rule leaves the two eyes with equal weights"
    assert abs(w_oja[:n].mean() - w_oja[n:].mean()) < 0.01

test_subtractive()
test_oja()
print("All checks passed")

### Exercise: Learning from single patterns

Draw x = L z with z standard normal and L the Cholesky factor of C, compute y = w·x, and apply dw = y x minus its mean, then clip, for 20,000 samples with eta = 0.002.

In [ ]:
L = np.linalg.cholesky(C)
w_s = w0.copy(); history = []
for t in range(20000):
    ...
    if t % 200 == 0:
        history.append([w_s[:n].mean(), w_s[n:].mean()])
history = np.array(history)
print(f"after 20,000 samples: left eye {w_s[:n].mean():.2f}, right eye {w_s[n:].mean():.2f}")
plt.plot(np.arange(len(history)) * 200, history); plt.legend(["left eye", "right eye"]); plt.xlabel("sample"); plt.ylabel("mean weight"); plt.show()

In [ ]:
import numpy as np
def test_segregates():
    "sample-by-sample learning also gives one eye weights above 0.9 and the other below 0.1"
    a, b = w_s[:n].mean(), w_s[n:].mean()
    assert max(a, b) > 0.9 and min(a, b) < 0.1

test_segregates()
print("All checks passed")